# Health Insurance Premium Prediction Using Machine Learning

**Objective:** Build, evaluate, tune, and persist a supervised Machine Learning regression pipeline to estimate annual health insurance charges from personal, demographic, and lifestyle features.

> **Dataset Prerequisite:** Download `insurance.csv` from [Kaggle (Medical Cost Personal Datasets)](https://www.kaggle.com/datasets/mirichoi0218/insurance) and place it at `../data/insurance.csv` before running all cells.


## Step 1: Data Loading & Initial Inspection


In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path so src/ modules can be imported cleanly
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeRegressor

from src import DEFAULT_DATA_PATH, DEFAULT_MODEL_PATH, RANDOM_STATE
from src.data_preprocessing import (
    clean_dataset,
    inspect_dataset,
    load_dataset,
    split_features_target,
    split_train_test,
)
from src.eda import compute_eda_summary, run_eda
from src.feature_engineering import add_engineered_features
from src.model_evaluation import (
    compute_regression_metrics,
    extract_feature_importance,
    perform_error_analysis,
    plot_actual_vs_predicted,
    plot_feature_importance,
    plot_model_comparison,
    plot_residuals,
)
from src.model_training import (
    build_model_pipeline,
    evaluate_models_for_feature_set,
    get_candidate_models,
    get_param_grid,
)

sns.set_theme(style="whitegrid", palette="Set2")
%matplotlib inline


In [ ]:
# Load the dataset
df_raw = load_dataset(DEFAULT_DATA_PATH)
print("Dataset Shape:", df_raw.shape)
print("Column Names:", list(df_raw.columns))
print("\nFirst 5 Records:")
display(df_raw.head())
print("\nLast 5 Records:")
display(df_raw.tail())


In [ ]:
# Check data types, info(), and descriptive statistics
print("Data Types:")
print(df_raw.dtypes)
print("\nDataset Info:")
df_raw.info()
print("\nDescriptive Statistics:")
display(df_raw.describe(include="all"))


## Step 2: Exploratory Data Analysis (EDA)

We explore the target distribution (`charges`) and its relationships with `age`, `bmi`, `smoker`, `sex`, `region`, and `children`, followed by a correlation heatmap.


In [ ]:
# 1. Distribution of Insurance Charges
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(df_raw["charges"], kde=True, bins=35, color="#2b6cb0", ax=axes[0])
axes[0].set_title("Distribution of Annual Insurance Charges")
axes[0].set_xlabel("Charges ($)")

sns.boxplot(x=df_raw["charges"], color="#63b3ed", ax=axes[1])
axes[1].set_title("Box Plot of Insurance Charges")
axes[1].set_xlabel("Charges ($)")
plt.tight_layout()
plt.show()


**Observation:** Annual insurance charges are strongly right-skewed, with most policyholders concentrated in the lower-to-mid range and a long upper tail of high-cost claims.


In [ ]:
# 2. Age vs. Charges & 3. BMI vs. Charges (grouped by smoking status)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df_raw, x="age", y="charges", hue="smoker", alpha=0.75, ax=axes[0])
axes[0].set_title("Age vs. Insurance Charges")

sns.scatterplot(data=df_raw, x="bmi", y="charges", hue="smoker", alpha=0.75, ax=axes[1])
axes[1].axvline(30.0, color="gray", linestyle="--", label="BMI = 30 (Obesity)")
axes[1].set_title("BMI vs. Insurance Charges")
axes[1].legend()
plt.tight_layout()
plt.show()


**Observation:** Charges increase steadily with age across distinct bands. In the BMI scatter plot, non-smokers show a relatively flat relationship with BMI, whereas smokers experience a steep upward jump in charges once BMI reaches 30 or above—indicating a strong `smoker × bmi` interaction.


In [ ]:
# 4. Smoker, 5. Gender, 6. Region, and 7. Children vs. Charges
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.boxplot(data=df_raw, x="smoker", y="charges", hue="smoker", legend=False, ax=axes[0, 0])
axes[0, 0].set_title("Smoking Status vs. Charges")

sns.boxplot(data=df_raw, x="sex", y="charges", hue="sex", legend=False, ax=axes[0, 1])
axes[0, 1].set_title("Gender vs. Charges")

sns.boxplot(data=df_raw, x="region", y="charges", hue="region", legend=False, ax=axes[1, 0])
axes[1, 0].set_title("Region vs. Charges")

sns.boxplot(data=df_raw, x="children", y="charges", hue="children", legend=False, ax=axes[1, 1])
axes[1, 1].set_title("Number of Children vs. Charges")

plt.tight_layout()
plt.show()


**Observation:** Smoking status creates the largest separation in insurance charges. Gender, residential region, and number of children show much smaller differences in median charges compared to smoking status.


In [ ]:
# 8. Correlation Heatmap
corr_df = df_raw[["age", "bmi", "children", "charges"]].copy()
corr_df["smoker_binary"] = (df_raw["smoker"].str.lower() == "yes").astype(int)

plt.figure(figsize=(7, 5.5))
sns.heatmap(corr_df.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


**Observation:** `smoker_binary` exhibits the strongest positive correlation with `charges`, followed by `age` and `bmi`, while `children` has a weak linear correlation.


## Step 3: Data Cleaning

We inspect missing values, remove exact duplicate rows, standardize categorical strings, and audit outliers using the 1.5 × IQR rule. High-charge records are retained because they represent genuine high-risk individuals rather than measurement errors.


In [ ]:
df_clean, cleaning_report = clean_dataset(df_raw)
print("Cleaning Report:")
for key, value in cleaning_report.items():
    print(f"- {key}: {value}")


## Step 4 & Step 6: Separate Features/Target and Train-Test Split (80/20)

We split into 80% training and 20% testing (`random_state=42`) **before** fitting any encoders, scalers, or models to eliminate data leakage.


In [ ]:
X, y = split_features_target(df_clean)
X_train, X_test, y_train, y_test = split_train_test(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape:  {X_test.shape}")


## Step 5 & Step 7: Feature Engineering Comparison & Model Training

We train and compare four regression models:
1. **Linear Regression** — Interpretable linear baseline.
2. **Decision Tree Regressor** — Captures non-linear splits and interactions.
3. **Random Forest Regressor** — Bagged tree ensemble reducing variance.
4. **Gradient Boosting Regressor** — Sequential boosting ensemble minimizing residual errors.

Each model is wrapped in a leakage-free `Pipeline` (`InsuranceFeatureEngineer` → `ColumnTransformer` → Estimator) and compared **without** vs. **with** engineered features (`bmi_category`, `age_group`, `smoker_bmi_interaction`, `is_obese_smoker`).


In [ ]:
# Preview engineered features on a small slice of training data
display(add_engineered_features(X_train.head()))

# Evaluate all 4 models with raw features only vs. engineered features
raw_results, raw_pipelines = evaluate_models_for_feature_set(
    X_train, X_test, y_train, y_test, use_engineered_features=False
)
eng_results, eng_pipelines = evaluate_models_for_feature_set(
    X_train, X_test, y_train, y_test, use_engineered_features=True
)

fe_comparison = pd.concat([raw_results, eng_results], ignore_index=True)
display(fe_comparison)


## Step 8: Model Evaluation & Selection

We select the winning feature set and candidate model based on 5-fold cross-validation RMSE on training data, then inspect test-set MAE, MSE, RMSE, and R².


In [ ]:
use_engineered = float(eng_results["CV_RMSE"].min()) <= float(raw_results["CV_RMSE"].min())
selected_results = eng_results if use_engineered else raw_results
selected_pipelines = eng_pipelines if use_engineered else raw_pipelines

print(f"Selected feature set: {'With Engineered Features' if use_engineered else 'Raw Features Only'}")
display(selected_results)

best_candidate_name = str(selected_results.sort_values("CV_RMSE").iloc[0]["Model"])
print(f"Best candidate model based on 5-fold CV RMSE: {best_candidate_name}")


## Step 9: Hyperparameter Tuning (`GridSearchCV`)

We tune the top candidate model using 5-fold cross-validation strictly on the training set, then compare untuned vs. tuned performance on the untouched test set.


In [ ]:
base_estimator = get_candidate_models(RANDOM_STATE)[best_candidate_name]
tuning_pipeline = build_model_pipeline(base_estimator, use_engineered_features=use_engineered)
param_grid = get_param_grid(best_candidate_name)

grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    refit=True,
)
grid_search.fit(X_train, y_train)

untuned_pred = selected_pipelines[best_candidate_name].predict(X_test)
tuned_pred = grid_search.best_estimator_.predict(X_test)

untuned_metrics = compute_regression_metrics(y_test, untuned_pred)
tuned_metrics = compute_regression_metrics(y_test, tuned_pred)

print("Best Hyperparameters:", grid_search.best_params_)
comparison_tuning = pd.DataFrame(
    [
        {"Stage": "Before Tuning", **untuned_metrics},
        {"Stage": "After Tuning", **tuned_metrics},
    ]
)
display(comparison_tuning)


In [ ]:
# Diagnostic visualizations & Error Analysis on the untouched test set
final_pipeline = grid_search.best_estimator_
y_final_pred = final_pipeline.predict(X_test)

# Actual vs. Predicted
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_final_pred, alpha=0.65, color="#2b6cb0", edgecolors="white", linewidth=0.4)
min_v, max_v = min(y_test.min(), y_final_pred.min()), max(y_test.max(), y_final_pred.max())
plt.plot([min_v, max_v], [min_v, max_v], "r--", label="Ideal (y = x)")
plt.title(f"Actual vs. Predicted Charges — {best_candidate_name} (Tuned)")
plt.xlabel("Actual Charges ($)")
plt.ylabel("Predicted Charges ($)")
plt.legend()
plt.tight_layout()
plt.show()

# Residual Plot
residuals = y_test - y_final_pred
plt.figure(figsize=(7, 4.5))
plt.scatter(y_final_pred, residuals, alpha=0.65, color="#3182ce")
plt.axhline(0.0, color="red", linestyle="--")
plt.title(f"Residuals vs. Predicted — {best_candidate_name} (Tuned)")
plt.xlabel("Predicted Charges ($)")
plt.ylabel("Residual ($)")
plt.tight_layout()
plt.show()

# Feature Importance
importance_df = extract_feature_importance(final_pipeline, top_n=12)
display(importance_df)

# Error Analysis
error_summary = perform_error_analysis(X_test, y_test, y_final_pred)
print("Error Analysis Summary:")
for k, v in error_summary.items():
    if k != "top_5_largest_errors":
        print(f"- {k}: {v}")
display(pd.DataFrame(error_summary["top_5_largest_errors"]))


## Step 10: Save & Reload Model Verification (`joblib`)

We save the complete preprocessing + prediction pipeline using `joblib` and verify that predictions match after reloading.


In [ ]:
DEFAULT_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(final_pipeline, DEFAULT_MODEL_PATH)
print(f"Saved complete pipeline to: {DEFAULT_MODEL_PATH}")

# Reload and verify prediction consistency
reloaded_pipeline = joblib.load(DEFAULT_MODEL_PATH)
sample_input = pd.DataFrame(
    [
        {
            "age": 35,
            "sex": "female",
            "bmi": 27.5,
            "children": 1,
            "smoker": "no",
            "region": "southeast",
        }
    ]
)

pred_original = float(final_pipeline.predict(sample_input)[0])
pred_reloaded = float(reloaded_pipeline.predict(sample_input)[0])
print(f"Original Pipeline Prediction: ${pred_original:,.2f}")
print(f"Reloaded Pipeline Prediction: ${pred_reloaded:,.2f}")
assert np.isclose(pred_original, pred_reloaded, rtol=1e-6), "Reload verification failed!"
print("Reload verification passed!")
